# 05 — Multi-Document Ingestion

In this notebook we will:
1. Use the reusable ingestion module (`src/ingestion.py`) to download multiple transcripts
2. Process 4 quarters of Apple earnings calls
3. Re-index everything in ChromaDB as a single collection
4. Verify we can query across quarters

### What changed from Phase 1
- The ingestion logic from notebook 01 has been extracted to `src/ingestion.py` so we can reuse it.
- We now process multiple transcripts in a loop instead of one at a time.
- ChromaDB will hold documents from all quarters, and metadata filtering lets us query specific ones.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")

import json
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions
from src.ingestion import process_transcript, save_transcript

print("Modules loaded")

Modules loaded


## Step 1: Define the transcripts to download

We'll process transcripts that use the new Motley Fool format (post-2025).

The new format uses `Name:` for speaker headers and `Role — Name` in the Call Participants section, instead of the old `Name\n--\nRole` format.

In [2]:
TRANSCRIPTS = [
    {
        "url": "https://www.fool.com/earnings/call-transcripts/2025/08/01/apple-aapl-q3-2025-earnings-call-transcript/",
        "company": "AAPL",
        "quarter": "Q3-2025",
        "date": "2025-07-31",
    },
    {
        "url": "https://www.fool.com/earnings/call-transcripts/2025/10/31/apple-q4-2025-earnings-call-transcript/",
        "company": "AAPL",
        "quarter": "Q4-2025",
        "date": "2025-10-30",
    },
    {
        "url": "https://www.fool.com/earnings/call-transcripts/2025/10/29/microsoft-msft-q1-2026-earnings-call-transcript/",
        "company": "MSFT",
        "quarter": "Q1-2026",
        "date": "2025-10-29",
    },
]

print(f"Transcripts to process: {len(TRANSCRIPTS)}")
for t in TRANSCRIPTS:
    print(f"  {t['company']} {t['quarter']} — {t['date']}")

Transcripts to process: 3
  AAPL Q3-2025 — 2025-07-31
  AAPL Q4-2025 — 2025-10-30
  MSFT Q1-2026 — 2025-10-29


## Step 2: Download and parse all transcripts

We use `process_transcript()` from `src/ingestion.py` — the same logic from notebook 01, now as a reusable function.

In [3]:
all_transcripts = []

for t in TRANSCRIPTS:
    print(f"Processing {t['quarter']}...", end=" ")
    transcript = process_transcript(
        url=t["url"],
        company=t["company"],
        quarter=t["quarter"],
        date=t["date"],
    )
    # Save to disk
    path = save_transcript(transcript, output_dir="../data/processed")
    all_transcripts.append(transcript)
    print(f"{transcript['total_turns']} turns — saved to {path}")

total_turns = sum(t["total_turns"] for t in all_transcripts)
print(f"\nTotal: {len(all_transcripts)} transcripts, {total_turns} turns")

Processing Q3-2025... 57 turns — saved to ../data/processed/AAPL_Q3_2025.json
Processing Q4-2025... 76 turns — saved to ../data/processed/AAPL_Q4_2025.json
Processing Q1-2026... 43 turns — saved to ../data/processed/MSFT_Q1_2026.json

Total: 3 transcripts, 176 turns


## Step 3: Build ChromaDB documents from all transcripts

Same logic as notebook 02, but now we loop over multiple transcripts.

---
### Your turn!

Build the three lists (`documents`, `metadatas`, `ids`) from all transcripts combined.

The challenge: IDs must be globally unique across all quarters. In notebook 02 you used an index — that won't work anymore since turn #5 from Q4-2024 and turn #5 from Q1-2025 would collide.

In [4]:
documents = []
metadatas = []
ids = []

for transcript in all_transcripts:
    for i, turn in enumerate(transcript['turns']):
        dict_transcript = {
            'company': transcript['company'], 'quarter': transcript['quarter'],
            'speaker': turn["speaker"], 'role': turn["role"],
        }
        turn_id = f"{transcript['company']}_{transcript['quarter']}_{i}"
        documents.append(turn["text"])
        metadatas.append(dict_transcript)
        ids.append(turn_id)

print(f"Documents: {len(documents)}")
print(f"Metadatas: {len(metadatas)}")
print(f"Unique IDs: {len(set(ids))}")
print(f"\nSample metadata: {metadatas[0]}")
print(f"Sample ID: {ids[0]}")

Documents: 176
Metadatas: 176
Unique IDs: 176

Sample metadata: {'company': 'AAPL', 'quarter': 'Q3-2025', 'speaker': 'Timothy D. Cook', 'role': 'Chief Executive Officer'}
Sample ID: AAPL_Q3-2025_0


## Step 4: Re-index in ChromaDB

In [5]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="../chroma_db")

# Delete old collection and recreate with all documents
try:
    client.delete_collection("earnings_calls")
except Exception:
    pass

collection = client.create_collection(
    name="earnings_calls",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

# Add in batches (ChromaDB recommends batches for large datasets)
BATCH_SIZE = 50
for i in range(0, len(documents), BATCH_SIZE):
    batch_end = min(i + BATCH_SIZE, len(documents))
    collection.add(
        documents=documents[i:batch_end],
        metadatas=metadatas[i:batch_end],
        ids=ids[i:batch_end],
    )
    print(f"  Added batch {i//BATCH_SIZE + 1}: documents {i+1}-{batch_end}")

print(f"\nTotal documents in collection: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Added batch 1: documents 1-50
  Added batch 2: documents 51-100
  Added batch 3: documents 101-150
  Added batch 4: documents 151-176

Total documents in collection: 176


## Step 5: Verify cross-quarter queries

In [6]:
def show_results(results, max_text=150):
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists)):
        print(f"--- Result {i+1} (distance: {dist:.3f}) ---")
        print(f"  Quarter: {meta.get('quarter')} | Speaker: {meta.get('speaker')} | Role: {meta.get('role')}")
        print(f"  Text: {doc[:max_text]}...")
        print()

In [7]:
# Query across all quarters
results = collection.query(
    query_texts=["What were the total revenue results?"],
    n_results=4,
)
print("Query: 'What were the total revenue results?' (all quarters)\n")
show_results(results)

Query: 'What were the total revenue results?' (all quarters)

--- Result 1 (distance: 0.422) ---
  Quarter: Q3-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Thanks, Tim, and good afternoon, everyone. Our revenue of $94 billion was up 10% year-over-year and is a new June quarter record. We grew in every geo...

--- Result 2 (distance: 0.452) ---
  Quarter: Q4-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Thanks, Tim, and good afternoon, everyone. Our revenue of $102.5 billion was up 8% year-over-year and is a new September quarter record. We set some t...

--- Result 3 (distance: 0.474) ---
  Quarter: Q4-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Okay. Yes, there was no tax-related impact. And what I would say is our strong performance for the quarter is really organically driven. And again, ju...

--- Result 4 (distance: 0.483) ---
  Quarter: Q4-2025 | Speaker: Timothy Cook | Role: 
  Text: Thank you, Suhasini. Goo

In [13]:
# Query filtered to a specific quarter
results_q4 = collection.query(
    query_texts=["What were the revenue results?"],
    n_results=3,
    where={"quarter": "Q4-2025"},
)
print("Query: 'What were the revenue results?' (Q4-2025 only)\n")
show_results(results_q4)

Query: 'What were the revenue results?' (Q4-2025 only)

--- Result 1 (distance: 0.468) ---
  Quarter: Q4-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Thanks, Tim, and good afternoon, everyone. Our revenue of $102.5 billion was up 8% year-over-year and is a new September quarter record. We set some t...

--- Result 2 (distance: 0.470) ---
  Quarter: Q4-2025 | Speaker: Kevan Parekh | Role: Chief Financial Officer
  Text: Okay. Yes, there was no tax-related impact. And what I would say is our strong performance for the quarter is really organically driven. And again, ju...

--- Result 3 (distance: 0.504) ---
  Quarter: Q4-2025 | Speaker: Timothy Cook | Role: 
  Text: Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Today, Apple is proud to report $102.5 billion in revenue, up 8% from ...



In [14]:
# Summary of what's in the collection
all_docs = collection.get(include=["metadatas"])
quarters = {}
for m in all_docs["metadatas"]:
    q = m.get("quarter", "unknown")
    quarters[q] = quarters.get(q, 0) + 1

print("Collection summary:")
print(f"  Total documents: {collection.count()}")
for q in sorted(quarters.keys()):
    print(f"  {q}: {quarters[q]} turns")

Collection summary:
  Total documents: 176
  Q1-2026: 43 turns
  Q3-2025: 57 turns
  Q4-2025: 76 turns


## Summary

In this notebook you learned:
- How to extract notebook logic into reusable Python modules
- How to process multiple documents in a loop
- The importance of globally unique IDs when combining multiple sources
- How to batch-insert into ChromaDB
- Cross-quarter queries work because metadata filtering operates on the `quarter` field

**Next step:** In notebook 06 we'll build advanced retrieval with `SelfQueryRetriever` — the LLM will automatically extract filters from natural language questions.